# S2 Combined Agreement — Yes/No + Free-Text (Unified Metric)

Integrates agreement scoring across all 113 questions by combining:
- **Yes/No (25 qids):** exact match + Cohen's κ (chance-corrected binary)
- **Free-text (88 qids):** clipped SBERT cosine (`sbert_score_clip`)

Then builds a single per-question HH / MM / HM agreement table and compares
to free-text-only results from `10_align_agreement.ipynb`.

**Why combine?**  
Free-text SBERT inflates semantically-related antonyms ("yes" vs "no" = 0.70),  
making it a poor discriminator for binary questions. Exact match + κ is more
appropriate there, but exact match is too strict for open-ended answers.  
The combined metric uses the best tool for each question type.

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

BASE    = Path('/home/david/Desktop/yuna/HPA')
EXPORTS = BASE / 'analysis/session2/exports'
sys.path.insert(0, str(BASE / 'analysis'))

from utils.constants import VARIANT_ORDER, VARIANT_LABELS, VARIANT_COLORS, GROUP_COLORS

In [ ]:
# ── Load yes/no pair data ────────────────────────────────────────────────────
# Built by 10b_align_agreement_yesno.ipynb
# Columns: question_id, variant, pair_type, answer_1, answer_2, exact_match
yn_pairs = pd.read_csv(EXPORTS / 'answer_pairs_yesno.csv')
yn_qids  = set(yn_pairs['question_id'].unique())
print(f'Yes/no pairs: {len(yn_pairs):,} rows, {len(yn_qids)} questions, '
      f'{yn_pairs["variant"].value_counts().to_dict()}')

# ── Load free-text pair data ─────────────────────────────────────────────────
# Built by 10_align_agreement.ipynb
ft_pairs = pd.read_parquet(EXPORTS / 'pair_cache.parquet')
ft_qids  = set(ft_pairs['question_id'].unique())
print(f'Free-text pairs: {len(ft_pairs):,} rows, {len(ft_qids)} questions, '
      f'{ft_pairs["variant"].value_counts().to_dict()}')
print(f'\nOverlap (should be 0): {len(yn_qids & ft_qids)}')
print(f'Total questions: {len(yn_qids | ft_qids)}')

In [ ]:
# ── Load human accuracy and question metadata ────────────────────────────────
h_csv = pd.read_csv(EXPORTS / 'responses_human.csv')
m_csv = pd.read_csv(EXPORTS / 'responses_model_inst_blind.csv')

q_meta = json.load(open(BASE / 'experiment/s2_v4/s4_question.json'))
q_en_map = {q['question_id']: q['question_en'] for q in q_meta}
answer_type_map = {q['question_id']: q['answer_type'] for q in q_meta}

# Human average accuracy per question (variant C)
h_acc_c = (h_csv[h_csv['variant'] == 'C']
           .groupby('question_id')['human_avg_acc'].mean())

print(f'Human data: {h_csv["participant"].nunique()} participants, '
      f'{h_csv["question_id"].nunique()} questions')
print(f'Human accuracy (variant C): mean={h_acc_c.mean():.3f}, '
      f'n={len(h_acc_c)} questions')

In [ ]:
# ── Cohen's kappa for yes/no agreement ──────────────────────────────────────
# For each (question_id, variant, pair_type):
#   p_obs  = fraction of pairs with exact match (same label)
#   yes_rate = fraction of answers that are 'yes' (positivity bias signal)
#   kappa = (p_obs - p_expected) / (1 - p_expected)
#         where p_expected = yes_rate^2 + (1-yes_rate)^2  (binary agreement by chance)
#
# Kappa is computed PER QUESTION, then aggregated (mean ± std across questions).
# NB: For HH, yes_rate from human answers only;
#     for HM, subject_1 is always human and subject_2 is always model.

def _yn_norm(text):
    t = str(text).strip().lower()
    if t.startswith('yes'): return 'yes'
    if t.startswith('no'):  return 'no'
    return 'other'

def compute_yn_question_metrics(pairs_df, answer_col_1='answer_1', answer_col_2='answer_2'):
    """Per-question agreement metrics from pair-level data."""
    rows = []
    for (qid, var, ptype), grp in pairs_df.groupby(['question_id', 'variant', 'pair_type']):
        a1 = [_yn_norm(a) for a in grp[answer_col_1]]
        a2 = [_yn_norm(a) for a in grp[answer_col_2]]
        n  = len(a1)
        if n == 0:
            continue

        exact = grp['exact_match'].mean()   # already computed

        # yes-rate from both sides combined
        all_a = a1 + a2
        yes_rate = sum(1 for a in all_a if a == 'yes') / max(len(all_a), 1)

        # Cohen's kappa (binary): chance = yes_rate^2 + no_rate^2
        p_exp = yes_rate ** 2 + (1 - yes_rate) ** 2
        kappa = (exact - p_exp) / (1 - p_exp) if (1 - p_exp) > 1e-9 else 0.0

        rows.append({
            'question_id': qid,
            'variant':     var,
            'pair_type':   ptype,
            'n_pairs':     n,
            'exact':       round(exact, 4),
            'yes_rate':    round(yes_rate, 4),
            'kappa':       round(kappa, 4),
        })
    return pd.DataFrame(rows)

yn_q_long = compute_yn_question_metrics(yn_pairs)

# Pivot to wide: one row per (question_id, variant)
yn_q = yn_q_long.pivot_table(
    index=['question_id', 'variant'],
    columns='pair_type',
    values=['exact', 'kappa', 'yes_rate'],
)
yn_q.columns = [f'{pt}_{m}' for m, pt in yn_q.columns]
yn_q = yn_q.reset_index()
yn_q['answer_type'] = 'yesno'

print(f'Yes/no question-level: {len(yn_q)} rows  (one per question × variant)')
print(f'Each kappa value = per-question κ; reported stats = mean ± std across {yn_q["question_id"].nunique()} questions\n')
C_yn = yn_q[yn_q['variant'] == 'C']
for m in ('exact', 'kappa'):
    parts = {pt: f'{C_yn[f"{pt}_{m}"].mean():.3f} ± {C_yn[f"{pt}_{m}"].std():.3f}'
             for pt in ('HH', 'MM', 'HM')}
    print(f'  {m}: ', parts)

In [ ]:
# ── Free-text question-level HH/MM/HM (from pair cache) ─────────────────────
# Primary metric: sbert_score_clip (clipped cosine, most reliable for free-text)
# Also include: simcse_score_clip, exact_score for comparison

ft_agg = (
    ft_pairs
    .groupby(['question_id', 'variant', 'pair_type'])
    [['sbert_score_clip', 'simcse_score_clip', 'exact_score', 'bertscore_f1']]
    .mean()
    .reset_index()
)

# Pivot
ft_q = ft_agg.pivot_table(
    index=['question_id', 'variant'],
    columns='pair_type',
    values=['sbert_score_clip', 'simcse_score_clip', 'exact_score', 'bertscore_f1'],
)
ft_q.columns = [f'{pt}_{m}' for m, pt in ft_q.columns]
ft_q = ft_q.reset_index()

# Add question text + answer_type
ft_q['question_en']  = ft_q['question_id'].map(q_en_map)
ft_q['answer_type']  = 'text'

print(f'Free-text question-level: {len(ft_q)} rows (question × variant)')
C_ft = ft_q[ft_q['variant']=='C']
for m in ('sbert_score_clip', 'exact_score'):
    print(f'  {m}: ', {pt: f'{C_ft[f"{pt}_{m}"].mean():.3f}' for pt in ('HH','MM','HM')})

In [ ]:
# ── Build unified per-question agreement table ────────────────────────────────
# One row per (question_id, variant) for all 113 questions.
# For yesno: primary metric = kappa (chance-corrected) + exact as backup
# For text:  primary metric = sbert_score_clip
# Unified HH / MM / HM columns: use primary metric appropriate for answer_type

# Merge yes/no into full format (add sbert columns as NaN)
yn_q['question_en'] = yn_q['question_id'].map(q_en_map)

# For yes/no: set HH/MM/HM to kappa
for pt in ('HH', 'MM', 'HM'):
    yn_q[f'{pt}_primary'] = yn_q[f'{pt}_kappa']

# For free-text: set HH/MM/HM to sbert_score_clip
for pt in ('HH', 'MM', 'HM'):
    ft_q[f'{pt}_primary'] = ft_q[f'{pt}_sbert_score_clip']

# Combine
shared_cols = ['question_id', 'question_en', 'variant', 'answer_type',
               'HH_primary', 'MM_primary', 'HM_primary']
yn_shared = yn_q[shared_cols + ['HH_exact', 'HM_exact', 'MM_exact',
                                'HH_kappa', 'HM_kappa', 'MM_kappa',
                                'HH_yes_rate', 'HM_yes_rate', 'MM_yes_rate']].copy()
ft_shared = ft_q[shared_cols + ['HH_sbert_score_clip', 'HM_sbert_score_clip', 'MM_sbert_score_clip',
                                 'HH_exact_score', 'HM_exact_score', 'MM_exact_score']].copy()

combined = pd.concat([yn_shared, ft_shared], ignore_index=True, sort=False)
combined = combined.sort_values(['question_id', 'variant']).reset_index(drop=True)

print(f'Combined table: {len(combined)} rows, {combined["question_id"].nunique()} questions')
print(f'By type: {combined["answer_type"].value_counts().to_dict()}')
print(f'By variant: {combined["variant"].value_counts().to_dict()}')

# Summary
C_all = combined[combined['variant']=='C']
print('\nMean HH/MM/HM by answer type (variant C):')
for atype in ('yesno', 'text'):
    sub = C_all[C_all['answer_type']==atype]
    hh = sub['HH_primary'].mean()
    mm = sub['MM_primary'].mean()
    hm = sub['HM_primary'].mean()
    print(f'  [{atype:6s}] HH={hh:.3f}  MM={mm:.3f}  HM={hm:.3f}  (n={len(sub)}q)')

# Overall (raw mean across all 113q — metric is not on same scale, so informational)
hh_all = C_all['HH_primary'].mean()
mm_all = C_all['MM_primary'].mean()
hm_all = C_all['HM_primary'].mean()
print(f'  [all 113q] HH={hh_all:.3f}  MM={mm_all:.3f}  HM={hm_all:.3f}')

In [ ]:
# ── Yes/no: exact match vs Cohen's κ comparison ───────────────────────────────
# Shows why kappa is more informative: many questions have high exact match
# simply because humans tend to say yes (positivity bias).

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

C_yn_full = yn_q_long[yn_q_long['variant'] == 'C']

# Left: per-question exact vs kappa scatter (HH)
ax = axes[0]
hh_q = C_yn_full[C_yn_full['pair_type'] == 'HH']
ax.scatter(hh_q['exact'], hh_q['kappa'], alpha=0.7, c='#333333', s=40)
ax.axhline(0, color='red', ls='--', lw=1.5, label='κ = 0 (chance level)')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3, label='exact = kappa')
ax.set_xlabel('Exact match (raw)')
ax.set_ylabel("Cohen's κ")
ax.set_title('HH: Exact vs κ (yes/no, variant C)')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# Middle: yes-rate distribution (HH) — positivity bias
ax = axes[1]
ax.hist(hh_q['yes_rate'], bins=15, color='#2196F3', alpha=0.75, edgecolor='white')
ax.axvline(0.5, color='red', ls='--', lw=2, label='50/50 (no bias)')
ax.axvline(hh_q['yes_rate'].mean(), color='#FF9800', ls='-', lw=2,
           label=f'mean={hh_q["yes_rate"].mean():.2f}')
ax.set_xlabel('Yes-rate (fraction of raters saying yes)')
ax.set_ylabel('Number of questions')
ax.set_title('Human yes-rate distribution\n(yes/no Qs, variant C)')
ax.legend(fontsize=8)

# Right: kappa by pair type (HH/MM/HM)
ax = axes[2]
colors_bt = {'HH': '#333333', 'MM': '#9C27B0', 'HM': '#FF5722'}
for i, pt in enumerate(['HH', 'MM', 'HM'], 1):
    sub = C_yn_full[C_yn_full['pair_type'] == pt]['kappa'].dropna()
    ax.boxplot(sub, positions=[i], widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=colors_bt[pt], alpha=0.7),
               medianprops=dict(color='white', lw=2))
    ax.scatter(np.random.normal(i, 0.05, size=len(sub)), sub,
               color=colors_bt[pt], s=22, alpha=0.7, zorder=3)
ax.axhline(0, color='red', ls='--', lw=1.5, label='κ = 0 (chance)')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['HH', 'MM', 'HM'])
ax.set_ylabel("Cohen's κ")
ax.set_title("Cohen's κ by pair type\n(yes/no, variant C)")
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.suptitle("Yes/No Agreement: Exact Match vs Cohen's κ (chance-corrected)", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# Summary: mean ± std across questions (per-question κ aggregated)
print("\nSummary (variant C, yes/no) — mean ± std across questions:")
agg = C_yn_full.groupby('pair_type')[['exact', 'kappa', 'yes_rate']].agg(['mean', 'std']).round(3)
agg.columns = ['_'.join(c) for c in agg.columns]
for pt in ['HH', 'MM', 'HM']:
    row = agg.loc[pt]
    print(f"  {pt}:  exact={row['exact_mean']:.3f}±{row['exact_std']:.3f}  "
          f"κ={row['kappa_mean']:.3f}±{row['kappa_std']:.3f}  "
          f"yes_rate={row['yes_rate_mean']:.3f}±{row['yes_rate_std']:.3f}")

In [ ]:
# ── Distribution: kappa (yes/no) vs sbert_clip (free-text) ───────────────────
# Shows scale difference + how both behave across pair types

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey='row')
pair_types = ['HH', 'MM', 'HM']
colors_bt = {'HH': '#333333', 'MM': '#9C27B0', 'HM': '#FF5722'}

# Row 0: kappa (yes/no)
C_yn_f = yn_q_long[yn_q_long['variant']=='C']
for j, pt in enumerate(pair_types):
    ax = axes[0][j]
    vals = C_yn_f[C_yn_f['pair_type']==pt]['kappa'].dropna()
    ax.hist(vals, bins=12, color=colors_bt[pt], alpha=0.75, edgecolor='white')
    ax.axvline(vals.mean(), color='black', lw=2, label=f'mean={vals.mean():.3f}')
    ax.axvline(0, color='red', ls='--', lw=1.5, label='κ=0')
    ax.set_title(f'{pt} — κ (yes/no, n={len(vals)}q)')
    ax.legend(fontsize=7)
    ax.set_xlabel("Cohen's κ")
    if j == 0: ax.set_ylabel('Count')

# Row 1: SBERT clip (free-text)
C_ft_f = ft_q[ft_q['variant']=='C']
for j, pt in enumerate(pair_types):
    ax = axes[1][j]
    col = f'{pt}_sbert_score_clip'
    vals = C_ft_f[col].dropna()
    ax.hist(vals, bins=14, color=colors_bt[pt], alpha=0.75, edgecolor='white')
    ax.axvline(vals.mean(), color='black', lw=2, label=f'mean={vals.mean():.3f}')
    ax.set_title(f'{pt} — SBERT clip (free-text, n={len(vals)}q)')
    ax.legend(fontsize=7)
    ax.set_xlabel('Clipped SBERT cosine')
    if j == 0: ax.set_ylabel('Count')

plt.suptitle('Agreement Distribution: κ (yes/no) vs Clipped SBERT (free-text)\nVariant C', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Combined HH / MM / HM across all 113 questions ───────────────────────────
# Side-by-side: yes/no (kappa) left, free-text (SBERT clip) right
# Shows structural differences in how humans and models agree for each question type

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

C_all_c = combined[combined['variant']=='C']
C_yn_c  = C_all_c[C_all_c['answer_type']=='yesno']
C_ft_c  = C_all_c[C_all_c['answer_type']=='text']

# Left: yes/no kappa
ax = axes[0]
for i, pt in enumerate(['HH', 'MM', 'HM'], 1):
    col = f'{pt}_kappa'
    vals = C_yn_c[col].dropna().values
    bp = ax.boxplot(vals, positions=[i], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor=colors_bt[pt], alpha=0.7),
                    medianprops=dict(color='white', lw=2), showfliers=False)
    x = np.random.normal(i, 0.06, size=len(vals))
    ax.scatter(x, vals, color=colors_bt[pt], s=28, alpha=0.7, zorder=3)
    ax.text(i, 0.55, f'μ={vals.mean():.3f}', ha='center', fontsize=8)
ax.axhline(0, color='red', ls='--', lw=1.5, label='κ = 0 (chance)')
ax.set_xticks([1,2,3])
ax.set_xticklabels(['HH\n(human–human)', 'MM\n(model–model)', 'HM\n(human–model)'])
ax.set_ylabel("Cohen's κ")
ax.set_title(f"Yes/No: Cohen's κ (n={C_yn_c['question_id'].nunique()}q, variant C)")
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Right: free-text sbert
ax = axes[1]
for i, pt in enumerate(['HH', 'MM', 'HM'], 1):
    col = f'{pt}_sbert_score_clip'
    vals = C_ft_c[col].dropna().values
    ax.boxplot(vals, positions=[i], widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=colors_bt[pt], alpha=0.7),
               medianprops=dict(color='white', lw=2), showfliers=False)
    x = np.random.normal(i, 0.06, size=len(vals))
    ax.scatter(x, vals, color=colors_bt[pt], s=28, alpha=0.7, zorder=3)
    ax.text(i, 0.85, f'μ={vals.mean():.3f}', ha='center', fontsize=8)
ax.set_xticks([1,2,3])
ax.set_xticklabels(['HH\n(human–human)', 'MM\n(model–model)', 'HM\n(human–model)'])
ax.set_ylabel('Clipped SBERT cosine')
ax.set_title(f'Free-text: Clipped SBERT (n={C_ft_c["question_id"].nunique()}q, variant C)')
ax.set_ylim(0, 1.02)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Combined Agreement: Yes/No (κ) + Free-Text (SBERT clip)\nVariant C, n=113 total questions',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── HM and MM agreement broken down by model group ───────────────────────────
# For HM pairs: model group = subject_group_2 (human is always subject_1)
# For MM pairs: same-group pairs only (subject_group_1 == subject_group_2)
# Cross-group MM pairs are excluded (ambiguous group label).
#
# Yes/no: kappa per question per group → mean ± std
# Free-text: sbert_score_clip per question per group → mean ± std

MODEL_GROUPS = ['VLM', 'VLM backbone decoder', 'standalone LLM']

colors_bt = {'HH': '#333333', 'MM': '#9C27B0', 'HM': '#FF5722'}
group_palette = {
    'VLM':                    GROUP_COLORS.get('VLM', '#2196F3'),
    'VLM backbone decoder':   GROUP_COLORS.get('VLM backbone decoder', '#4CAF50'),
    'standalone LLM':         GROUP_COLORS.get('standalone LLM', '#FF9800'),
}

# ── Yes/no: per-question kappa by group ──────────────────────────────────────
def _kappa_per_q(pairs_grp):
    """Given a subset of yn_pairs for one (qid, ptype, group), return per-question kappa."""
    rows = []
    for qid, g in pairs_grp.groupby('question_id'):
        exact = g['exact_match'].mean()
        all_a = [_yn_norm(a) for a in list(g['answer_1']) + list(g['answer_2'])]
        yr    = sum(1 for a in all_a if a == 'yes') / max(len(all_a), 1)
        p_exp = yr**2 + (1-yr)**2
        kappa = (exact - p_exp) / (1-p_exp) if (1-p_exp) > 1e-9 else 0.0
        rows.append({'question_id': qid, 'kappa': kappa, 'exact': exact, 'yes_rate': yr})
    return pd.DataFrame(rows)

C_yn_pairs = yn_pairs[yn_pairs['variant'] == 'C']

yn_group_rows = []
for ptype in ('HM', 'MM'):
    sub = C_yn_pairs[C_yn_pairs['pair_type'] == ptype]
    if ptype == 'HM':
        for grp in MODEL_GROUPS:
            g_pairs = sub[sub['subject_group_2'] == grp]
            if g_pairs.empty: continue
            q_df = _kappa_per_q(g_pairs)
            yn_group_rows.append({'pair_type': ptype, 'model_group': grp,
                                   'kappa_mean': q_df['kappa'].mean(),
                                   'kappa_std':  q_df['kappa'].std(),
                                   'exact_mean': q_df['exact'].mean(),
                                   'n_q':        len(q_df)})
    else:  # MM — same-group only
        for grp in MODEL_GROUPS:
            g_pairs = sub[(sub['subject_group_1'] == grp) & (sub['subject_group_2'] == grp)]
            if g_pairs.empty: continue
            q_df = _kappa_per_q(g_pairs)
            yn_group_rows.append({'pair_type': ptype, 'model_group': grp,
                                   'kappa_mean': q_df['kappa'].mean(),
                                   'kappa_std':  q_df['kappa'].std(),
                                   'exact_mean': q_df['exact'].mean(),
                                   'n_q':        len(q_df)})

yn_group_df = pd.DataFrame(yn_group_rows)

# ── Free-text: per-question sbert_score_clip by group ────────────────────────
C_ft_pairs = ft_pairs[ft_pairs['variant'] == 'C']

ft_group_rows = []
for ptype in ('HM', 'MM'):
    sub = C_ft_pairs[C_ft_pairs['pair_type'] == ptype]
    if ptype == 'HM':
        for grp in MODEL_GROUPS:
            g_pairs = sub[sub['subject_group_2'] == grp]
            if g_pairs.empty: continue
            q_means = g_pairs.groupby('question_id')['sbert_score_clip'].mean()
            ft_group_rows.append({'pair_type': ptype, 'model_group': grp,
                                   'sbert_mean': q_means.mean(),
                                   'sbert_std':  q_means.std(),
                                   'n_q':        len(q_means)})
    else:
        for grp in MODEL_GROUPS:
            g_pairs = sub[(sub['subject_group_1'] == grp) & (sub['subject_group_2'] == grp)]
            if g_pairs.empty: continue
            q_means = g_pairs.groupby('question_id')['sbert_score_clip'].mean()
            ft_group_rows.append({'pair_type': ptype, 'model_group': grp,
                                   'sbert_mean': q_means.mean(),
                                   'sbert_std':  q_means.std(),
                                   'n_q':        len(q_means)})

ft_group_df = pd.DataFrame(ft_group_rows)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

x     = np.arange(len(MODEL_GROUPS))
w     = 0.32
pt_colors = {'HM': '#FF5722', 'MM': '#9C27B0'}

for row_idx, (atype, group_df, metric, ylabel) in enumerate([
    ('yesno', yn_group_df, 'kappa', "Cohen's κ"),
    ('text',  ft_group_df, 'sbert', 'Clipped SBERT cosine'),
]):
    for col_idx, ptype in enumerate(['HM', 'MM']):
        ax = axes[row_idx][col_idx]
        sub = group_df[group_df['pair_type'] == ptype]

        bars_x, bars_h, bars_e, bar_labels = [], [], [], []
        for xi, grp in enumerate(MODEL_GROUPS):
            row = sub[sub['model_group'] == grp]
            if row.empty:
                bars_x.append(xi)
                bars_h.append(0)
                bars_e.append(0)
                bar_labels.append(f'{grp}\n(no data)')
            else:
                m = float(row[f'{metric}_mean'].iloc[0])
                s = float(row[f'{metric}_std'].iloc[0])
                n = int(row['n_q'].iloc[0])
                bars_x.append(xi)
                bars_h.append(m)
                bars_e.append(s)
                bar_labels.append(f'{grp}\n(n={n}q)')

        bar_colors = [group_palette.get(g, '#888888') for g in MODEL_GROUPS]
        ax.bar(x, bars_h, color=bar_colors, alpha=0.8, width=0.55, edgecolor='white', lw=0.5)
        ax.errorbar(x, bars_h, yerr=bars_e, fmt='none', color='black', capsize=5, lw=1.5)

        # Value labels
        for xi, (h, e) in enumerate(zip(bars_h, bars_e)):
            if h != 0:
                ax.text(xi, h + e + 0.01, f'{h:.3f}\n±{e:.3f}',
                        ha='center', va='bottom', fontsize=7.5)

        # HH reference line for context
        if atype == 'yesno':
            hh_ref = yn_q_long[(yn_q_long['variant']=='C') & (yn_q_long['pair_type']=='HH')]['kappa'].mean()
            ax.axhline(hh_ref, color='#333333', ls='--', lw=1.8,
                       label=f'HH ref={hh_ref:.3f}')
            ax.axhline(0, color='red', ls=':', lw=1.5, label='κ=0 (chance)')
        else:
            hh_ref = ft_pairs[(ft_pairs['variant']=='C') & (ft_pairs['pair_type']=='HH')].groupby('question_id')['sbert_score_clip'].mean().mean()
            ax.axhline(hh_ref, color='#333333', ls='--', lw=1.8,
                       label=f'HH ref={hh_ref:.3f}')

        ax.set_xticks(x)
        ax.set_xticklabels(bar_labels, fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ptype} — {"yes/no κ" if atype=="yesno" else "free-text SBERT"} by model group\n(variant C, error bars = ±1 SD across questions)')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

plt.suptitle('HM and MM Agreement by Model Group\n(MM = same-group pairs only; error bars = ±1 SD across questions)',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# ── Summary tables ────────────────────────────────────────────────────────────
from IPython.display import display
print('── Yes/No κ by group (variant C) ──')
display(yn_group_df.round(3))
print('\n── Free-text SBERT by group (variant C) ──')
display(ft_group_df.round(3))

In [ ]:
# ── Question-normalized group comparison ──────────────────────────────────────
# Raw scores (previous cell) mix two sources of variance:
#   1. Question difficulty  — some questions are hard/easy for EVERYONE
#   2. Group differences    — some groups are consistently closer to humans
#
# Normalization removes (1) by subtracting the per-question mean across groups,
# leaving only the relative group advantage/disadvantage on each question.
# This makes inter-group differences clearer but cannot be directly compared to HH.
#
# Method: for each question, compute group deviation = group_score - mean(all groups)
#   → positive = this group agrees MORE than average on this question
#   → negative = this group agrees LESS than average

# ── Yes/no: per-question kappa deviation by group ─────────────────────────────
C_yn_pairs = yn_pairs[yn_pairs['variant'] == 'C']

# Build (question_id × model_group) kappa table for HM and MM
def _build_kappa_table(pairs_df, ptype):
    """Returns DataFrame: index=question_id, columns=model_groups, values=kappa."""
    rows = {}
    sub = pairs_df[pairs_df['pair_type'] == ptype]
    grp_col = 'subject_group_2' if ptype == 'HM' else None
    for grp in MODEL_GROUPS:
        if ptype == 'HM':
            g_pairs = sub[sub['subject_group_2'] == grp]
        else:  # MM same-group
            g_pairs = sub[(sub['subject_group_1'] == grp) & (sub['subject_group_2'] == grp)]
        if g_pairs.empty:
            continue
        for qid, g in g_pairs.groupby('question_id'):
            exact = g['exact_match'].mean()
            all_a = [_yn_norm(a) for a in list(g['answer_1']) + list(g['answer_2'])]
            yr    = sum(1 for a in all_a if a == 'yes') / max(len(all_a), 1)
            p_exp = yr**2 + (1-yr)**2
            kappa = (exact - p_exp) / (1-p_exp) if (1-p_exp) > 1e-9 else 0.0
            rows.setdefault(qid, {})[grp] = kappa
    return pd.DataFrame(rows).T   # (n_questions × n_groups)

def _build_sbert_table(pairs_df, ptype):
    """Returns DataFrame: index=question_id, columns=model_groups, values=mean sbert_clip."""
    rows = {}
    sub = pairs_df[pairs_df['pair_type'] == ptype]
    for grp in MODEL_GROUPS:
        if ptype == 'HM':
            g_pairs = sub[sub['subject_group_2'] == grp]
        else:
            g_pairs = sub[(sub['subject_group_1'] == grp) & (sub['subject_group_2'] == grp)]
        if g_pairs.empty:
            continue
        for qid, g in g_pairs.groupby('question_id'):
            rows.setdefault(qid, {})[grp] = g['sbert_score_clip'].mean()
    return pd.DataFrame(rows).T

C_ft_pairs = ft_pairs[ft_pairs['variant'] == 'C']

tables = {
    ('yesno', 'HM'): _build_kappa_table(C_yn_pairs, 'HM'),
    ('yesno', 'MM'): _build_kappa_table(C_yn_pairs, 'MM'),
    ('text',  'HM'): _build_sbert_table(C_ft_pairs, 'HM'),
    ('text',  'MM'): _build_sbert_table(C_ft_pairs, 'MM'),
}

# Normalize: subtract row mean (= per-question mean across groups)
norm_tables = {}
for key, tbl in tables.items():
    row_mean = tbl.mean(axis=1)             # mean across groups for each question
    norm_tables[key] = tbl.sub(row_mean, axis=0)

# ── Plot: raw vs normalized side by side ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

metric_info = [
    ('yesno', 'kappa', "Cohen's κ"),
    ('text',  'sbert', 'Clipped SBERT'),
]

for row_idx, (atype, metric_short, ylabel) in enumerate(metric_info):
    for col_offset, ptype in enumerate(['HM', 'MM']):
        raw_tbl  = tables[(atype, ptype)]
        norm_tbl = norm_tables[(atype, ptype)]
        grp_cols = [g for g in MODEL_GROUPS if g in raw_tbl.columns]

        # Raw (left of pair)
        ax_raw  = axes[row_idx][col_offset * 2]
        ax_norm = axes[row_idx][col_offset * 2 + 1]

        for ax, tbl, title_suffix, zero_ref in [
            (ax_raw,  raw_tbl,  'raw',        False),
            (ax_norm, norm_tbl, 'normalized', True),
        ]:
            means = [tbl[g].mean() for g in grp_cols]
            stds  = [tbl[g].std()  for g in grp_cols]
            xs    = np.arange(len(grp_cols))
            colors = [group_palette.get(g, '#888') for g in grp_cols]

            ax.bar(xs, means, color=colors, alpha=0.8, width=0.55,
                   edgecolor='white', lw=0.5)
            ax.errorbar(xs, means, yerr=stds, fmt='none',
                        color='black', capsize=5, lw=1.5)
            for xi, (m, s) in enumerate(zip(means, stds)):
                ax.text(xi, m + s + (0.005 if not zero_ref else 0.003),
                        f'{m:+.3f}\n±{s:.3f}', ha='center', va='bottom', fontsize=7)

            ax.axhline(0, color='red' if zero_ref else '#333333',
                       ls='--', lw=1.5,
                       label='group mean' if zero_ref else
                             f'HH={yn_q_long[(yn_q_long.variant=="C")&(yn_q_long.pair_type=="HH")]["kappa"].mean():.3f}'
                             if atype == 'yesno' else
                             f'HH={C_ft_pairs[C_ft_pairs.pair_type=="HH"].groupby("question_id")["sbert_score_clip"].mean().mean():.3f}')
            ax.set_xticks(xs)
            short_labels = [g.replace('VLM backbone decoder', 'VLM-LM')
                              .replace('standalone LLM', 'LLM') for g in grp_cols]
            ax.set_xticklabels(short_labels, fontsize=8)
            ax.set_ylabel(ylabel if col_offset == 0 else '')
            ax.set_title(f'{ptype} {atype} — {title_suffix}\n(n={len(tbl)}q, ±1 SD)')
            ax.legend(fontsize=7)
            ax.grid(axis='y', alpha=0.3)

plt.suptitle(
    'Raw vs Question-Normalized Group Agreement\n'
    'Normalized = group score − per-question mean across groups (removes question difficulty)',
    y=1.02, fontsize=12
)
plt.tight_layout()
plt.show()

# ── Per-question deviation table ──────────────────────────────────────────────
print('── HM free-text: per-question SBERT deviation from group mean ──')
norm_ft_hm = norm_tables[('text', 'HM')].copy()
norm_ft_hm.index = norm_ft_hm.index.map(q_en_map)
display = __import__('IPython.display', fromlist=['display']).display
display(norm_ft_hm[[g for g in MODEL_GROUPS if g in norm_ft_hm.columns]].round(3).sort_values(
    [g for g in MODEL_GROUPS if g in norm_ft_hm.columns][0]))

In [ ]:
# ── LOO-normalized group comparison ───────────────────────────────────────────
# Reference scale: per-question leave-one-out (LOO) human agreement.
#
# For each question q, for each human rater i:
#   LOO_score(i, q) = mean agreement of rater i with all other human raters on q
# This gives a distribution of n_humans scores per question.
#   LOO_min(q) = worst human rater's mean agreement on q
#   LOO_max(q) = best human rater's mean agreement on q
#
# Normalized score = (raw_score - LOO_min(q)) / (LOO_max(q) - LOO_min(q))
#   0   = performs at worst-human level on this question
#   1   = performs at best-human level on this question
#   >1  = exceeds best human (model is more consistent than any single human)
#   <0  = below worst human
#
# This normalization is question-specific and human-anchored — not circular,
# since HM scores involve human–model pairs, not the human–human pairs used here.

# ── Build LOO reference ranges from HH pairs ──────────────────────────────────
ft_hh = ft_pairs[(ft_pairs['variant'] == 'C') & (ft_pairs['pair_type'] == 'HH')]
yn_hh = yn_pairs[(yn_pairs['variant'] == 'C') & (yn_pairs['pair_type'] == 'HH')]

def _loo_range_ft(qid):
    """Per-rater mean SBERT clip → (min, max, {rater: score})."""
    q = ft_hh[ft_hh['question_id'] == qid]
    rater_scores = {}
    for _, row in q.iterrows():
        rater_scores.setdefault(row['subject_1'], []).append(row['sbert_score_clip'])
        rater_scores.setdefault(row['subject_2'], []).append(row['sbert_score_clip'])
    if len(rater_scores) < 2:
        return np.nan, np.nan, {}
    per_rater = {r: np.mean(v) for r, v in rater_scores.items()}
    return min(per_rater.values()), max(per_rater.values()), per_rater

def _loo_range_yn(qid):
    """Per-rater mean exact match → (min, max, {rater: score})."""
    q = yn_hh[yn_hh['question_id'] == qid]
    rater_scores = {}
    for _, row in q.iterrows():
        rater_scores.setdefault(row['subject_1'], []).append(float(row['exact_match']))
        rater_scores.setdefault(row['subject_2'], []).append(float(row['exact_match']))
    if len(rater_scores) < 2:
        return np.nan, np.nan, {}
    per_rater = {r: np.mean(v) for r, v in rater_scores.items()}
    return min(per_rater.values()), max(per_rater.values()), per_rater

# Pre-compute LOO ranges
ft_loo = {qid: _loo_range_ft(qid) for qid in sorted(ft_qids)}
yn_loo = {qid: _loo_range_yn(qid) for qid in sorted(yn_qids)}

# ── Normalize HM group scores ─────────────────────────────────────────────────
def _norm(score, lo, hi):
    if np.isnan(lo) or np.isnan(hi) or (hi - lo) < 1e-9:
        return np.nan
    return (score - lo) / (hi - lo)

# Free-text HM: per-question per-group sbert → normalized
ft_hm_norm = {grp: {} for grp in MODEL_GROUPS}
ft_hm_raw  = {grp: {} for grp in MODEL_GROUPS}
C_ft_hm = C_ft_pairs[C_ft_pairs['pair_type'] == 'HM']
for grp in MODEL_GROUPS:
    g = C_ft_hm[C_ft_hm['subject_group_2'] == grp]
    for qid, qgrp in g.groupby('question_id'):
        raw = qgrp['sbert_score_clip'].mean()
        lo, hi, _ = ft_loo.get(qid, (np.nan, np.nan, {}))
        ft_hm_raw[grp][qid]  = raw
        ft_hm_norm[grp][qid] = _norm(raw, lo, hi)

# Yes/no HM: per-question per-group exact match → normalized
yn_hm_norm = {grp: {} for grp in MODEL_GROUPS}
yn_hm_raw  = {grp: {} for grp in MODEL_GROUPS}
C_yn_hm = C_yn_pairs[C_yn_pairs['pair_type'] == 'HM']
for grp in MODEL_GROUPS:
    g = C_yn_hm[C_yn_hm['subject_group_2'] == grp]
    for qid, qgrp in g.groupby('question_id'):
        raw = qgrp['exact_match'].mean()
        lo, hi, _ = yn_loo.get(qid, (np.nan, np.nan, {}))
        yn_hm_raw[grp][qid]  = raw
        yn_hm_norm[grp][qid] = _norm(raw, lo, hi)

# Also normalize the LOO per-rater HH scores themselves (by definition in [0,1] on average)
ft_loo_rater_scores = {}   # qid → list of per-rater normalized scores
for qid in sorted(ft_qids):
    lo, hi, per_rater = ft_loo[qid]
    if np.isnan(lo): continue
    ft_loo_rater_scores[qid] = [_norm(s, lo, hi) for s in per_rater.values()]

yn_loo_rater_scores = {}
for qid in sorted(yn_qids):
    lo, hi, per_rater = yn_loo[qid]
    if np.isnan(lo): continue
    yn_loo_rater_scores[qid] = [_norm(s, lo, hi) for s in per_rater.values()]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, atype, norm_dict, loo_rater, ylabel_raw in [
    (axes[0], 'free-text', ft_hm_norm, ft_loo_rater_scores, 'SBERT clip'),
    (axes[1], 'yes/no',    yn_hm_norm, yn_loo_rater_scores, 'exact match'),
]:
    grp_cols = [g for g in MODEL_GROUPS if norm_dict[g]]
    xs = np.arange(len(grp_cols))

    # HH LOO distribution: all per-rater normalized scores across all questions
    all_loo = [s for scores in loo_rater.values() for s in scores]
    loo_mean = np.nanmean(all_loo)
    loo_std  = np.nanstd(all_loo)

    # Per-group: collect normalized scores across questions
    grp_means, grp_stds, grp_labels = [], [], []
    for grp in grp_cols:
        vals = [v for v in norm_dict[grp].values() if not np.isnan(v)]
        grp_means.append(np.mean(vals) if vals else np.nan)
        grp_stds.append(np.std(vals)   if vals else np.nan)
        grp_labels.append(grp.replace('VLM backbone decoder', 'VLM-LM')
                             .replace('standalone LLM', 'LLM'))

    colors = [group_palette.get(g, '#888') for g in grp_cols]
    ax.bar(xs, grp_means, color=colors, alpha=0.8, width=0.55,
           edgecolor='white', lw=0.5)
    ax.errorbar(xs, grp_means, yerr=grp_stds, fmt='none',
                color='black', capsize=6, lw=1.8)
    for xi, (m, s) in enumerate(zip(grp_means, grp_stds)):
        if not np.isnan(m):
            ax.text(xi, m + s + 0.02, f'{m:+.3f}\n±{s:.3f}',
                    ha='center', va='bottom', fontsize=8)

    # HH LOO reference band: mean ± std of per-rater LOO normalized scores
    ax.axhline(loo_mean, color='#333333', ls='--', lw=2,
               label=f'HH LOO mean = {loo_mean:.3f}')
    ax.axhspan(loo_mean - loo_std, loo_mean + loo_std,
               alpha=0.12, color='#333333', label=f'HH ±1 SD = [{loo_mean-loo_std:.2f}, {loo_mean+loo_std:.2f}]')
    ax.axhline(0, color='blue', ls=':', lw=1.5, label='0 = worst human')
    ax.axhline(1, color='green', ls=':', lw=1.5, label='1 = best human')

    ax.set_xticks(xs)
    ax.set_xticklabels(grp_labels, fontsize=9)
    ax.set_ylabel(f'LOO-normalized HM score\n(0 = worst human, 1 = best human)')
    ax.set_title(f'HM alignment — {atype} ({len(norm_dict[grp_cols[0]])}q)\nLOO-normalized by per-question human range')
    ax.legend(fontsize=7.5, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(
    'LOO-Normalized HM Agreement by Model Group\n'
    'Scale: 0 = worst human rater, 1 = best human rater on each question',
    y=1.02, fontsize=12
)
plt.tight_layout()
plt.show()

# ── Print summary ─────────────────────────────────────────────────────────────
print('LOO-normalized HM scores (variant C)\n'
      '  0 = worst human on that question, 1 = best human\n')
for atype, norm_dict in [('free-text', ft_hm_norm), ('yes/no', yn_hm_norm)]:
    print(f'  [{atype}]')
    for grp in MODEL_GROUPS:
        vals = [v for v in norm_dict[grp].values() if not np.isnan(v)]
        if vals:
            print(f'    {grp:30s}  mean={np.mean(vals):+.3f}  std={np.std(vals):.3f}  n={len(vals)}q')

In [ ]:
# ── Across variants: HM agreement trend C → B → A, multiple metrics ───────────
# Each metric shown with a distinct marker+linestyle; pair types (HH/MM/HM) as colors.
# Yes/no metrics: exact match + Cohen's κ
# Free-text metrics: sbert_score_clip + simcse_score_clip + exact_score
#
# Source data:
#   yn_q      : wide table (question_id × variant), cols = {HH,MM,HM}_{metric}
#   ft_q      : same for free-text
# Both already contain all three variants.

METRIC_STYLE = {
    # yes/no
    'kappa':            dict(marker='o', ls='-',  lw=2,   ms=7,  label='Cohen κ'),
    'exact':            dict(marker='^', ls='--', lw=1.5, ms=7,  label='Exact match'),
    # free-text
    'sbert_score_clip': dict(marker='o', ls='-',  lw=2,   ms=7,  label='SBERT clip'),
    'simcse_score_clip':dict(marker='^', ls='--', lw=1.5, ms=7,  label='SimCSE clip'),
    'exact_score':      dict(marker='s', ls=':',  lw=1.5, ms=6,  label='Exact match'),
}

colors_pt = {'HH': '#333333', 'MM': '#9C27B0', 'HM': '#FF5722'}
x = np.arange(len(VARIANT_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: yes/no ──────────────────────────────────────────────────────────────
ax = axes[0]
yn_metrics = ['kappa', 'exact']

for pt in ['HH', 'MM', 'HM']:
    for m_idx, metric in enumerate(yn_metrics):
        col = f'{pt}_{metric}'
        if col not in yn_q.columns:
            continue
        style = METRIC_STYLE[metric]
        means = [yn_q[yn_q['variant'] == v][col].mean() for v in VARIANT_ORDER]
        sems  = [yn_q[yn_q['variant'] == v][col].sem()  for v in VARIANT_ORDER]
        label = f'{pt} — {style["label"]}' if m_idx == 0 else f'— {style["label"]}'
        line, = ax.plot(x, means,
                        marker=style['marker'], ls=style['ls'],
                        lw=style['lw'], ms=style['ms'],
                        color=colors_pt[pt],
                        alpha=0.9 if pt == 'HM' else 0.55,
                        label=label)
        ax.fill_between(x,
                        np.array(means) - 1.96 * np.array(sems),
                        np.array(means) + 1.96 * np.array(sems),
                        color=colors_pt[pt], alpha=0.07)

ax.axhline(0, color='red', ls=':', lw=1.2, alpha=0.6, label='κ = 0 (chance)')
ax.set_xticks(x)
ax.set_xticklabels(['C (original)', 'B (weaker obj)', 'A (pronominalized)'])
ax.set_ylabel('Agreement score')
ax.set_title(f'Yes/No ({yn_q["question_id"].nunique()}q) — C→B→A\n'
             f'circle+solid = κ,  triangle+dash = exact  (±95% CI)')
ax.grid(axis='y', alpha=0.3)

# Compact legend: pair-type colors (left block) + metric markers (right block)
from matplotlib.lines import Line2D
legend_handles = (
    [Line2D([0],[0], color=colors_pt[pt], lw=2, label=pt) for pt in ['HH','MM','HM']] +
    [Line2D([0],[0], color='gray', marker=METRIC_STYLE[m]['marker'],
             ls=METRIC_STYLE[m]['ls'], lw=1.5, ms=6, label=METRIC_STYLE[m]['label'])
     for m in yn_metrics]
)
ax.legend(handles=legend_handles, fontsize=8, ncol=2)

# ── Right: free-text ──────────────────────────────────────────────────────────
ax = axes[1]
ft_metrics = ['sbert_score_clip', 'simcse_score_clip', 'exact_score']

for pt in ['HH', 'MM', 'HM']:
    for m_idx, metric in enumerate(ft_metrics):
        col = f'{pt}_{metric}'
        if col not in ft_q.columns:
            continue
        style = METRIC_STYLE[metric]
        means = [ft_q[ft_q['variant'] == v][col].mean() for v in VARIANT_ORDER]
        sems  = [ft_q[ft_q['variant'] == v][col].sem()  for v in VARIANT_ORDER]
        ax.plot(x, means,
                marker=style['marker'], ls=style['ls'],
                lw=style['lw'], ms=style['ms'],
                color=colors_pt[pt],
                alpha=0.9 if pt == 'HM' else 0.55)
        ax.fill_between(x,
                        np.array(means) - 1.96 * np.array(sems),
                        np.array(means) + 1.96 * np.array(sems),
                        color=colors_pt[pt], alpha=0.07)

ax.set_xticks(x)
ax.set_xticklabels(['C (original)', 'B (weaker obj)', 'A (pronominalized)'])
ax.set_ylabel('Agreement score')
ax.set_title(f'Free-text ({ft_q["question_id"].nunique()}q) — C→B→A\n'
             f'circle = SBERT,  triangle = SimCSE,  square = exact  (±95% CI)')
ax.grid(axis='y', alpha=0.3)

legend_handles = (
    [Line2D([0],[0], color=colors_pt[pt], lw=2, label=pt) for pt in ['HH','MM','HM']] +
    [Line2D([0],[0], color='gray', marker=METRIC_STYLE[m]['marker'],
             ls=METRIC_STYLE[m]['ls'], lw=1.5, ms=6, label=METRIC_STYLE[m]['label'])
     for m in ft_metrics]
)
ax.legend(handles=legend_handles, fontsize=8, ncol=2)

plt.suptitle('Agreement Trend Across Variants (C→B→A) — Multiple Metrics\n'
             'Colors = pair type (HH/MM/HM),  Markers = metric', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation: HM agreement vs human accuracy (variant C) ──────────────────
# Does higher human–model agreement predict questions where humans get it right?
# Separate panels for yes/no (kappa) and free-text (SBERT clip).

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, atype, col, xlabel in [
    (axes[0], 'yesno', 'HM_kappa',            "HM Cohen's κ"),
    (axes[1], 'text',  'HM_sbert_score_clip',  'HM Clipped SBERT cosine'),
]:
    sub = C_all_c[C_all_c['answer_type']==atype].copy()
    sub['human_acc'] = sub['question_id'].map(h_acc_c)
    sub = sub.dropna(subset=[col, 'human_acc'])
    
    r, p = pearsonr(sub[col], sub['human_acc'])
    rho, _ = spearmanr(sub[col], sub['human_acc'])
    
    ax.scatter(sub[col], sub['human_acc'], alpha=0.65, s=40, color='#2196F3', edgecolor='white')
    # Regression line
    m, b = np.polyfit(sub[col], sub['human_acc'], 1)
    xline = np.linspace(sub[col].min(), sub[col].max(), 100)
    ax.plot(xline, m*xline+b, 'r-', lw=2, alpha=0.8)
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Human accuracy (variant C)')
    ax.set_title(f'{atype.capitalize()} (n={len(sub)}q)\nr={r:.3f}, ρ={rho:.3f}, p={p:.3f}')
    ax.grid(alpha=0.2)

plt.suptitle('HM Agreement vs Human Accuracy (variant C)\nDo models agree more with humans on questions humans get right?',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── HH agreement vs human accuracy (variant C) ───────────────────────────────
# Companion: how much does human–human agreement predict human accuracy?
# High HH + low acc = question with systematic wrong bias (linguistic prior!)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, atype, col, xlabel in [
    (axes[0], 'yesno', 'HH_kappa',            "HH Cohen's κ"),
    (axes[1], 'text',  'HH_sbert_score_clip',  'HH Clipped SBERT cosine'),
]:
    sub = C_all_c[C_all_c['answer_type']==atype].copy()
    sub['human_acc'] = sub['question_id'].map(h_acc_c)
    sub = sub.dropna(subset=[col, 'human_acc'])

    r, p = pearsonr(sub[col], sub['human_acc'])
    rho, _ = spearmanr(sub[col], sub['human_acc'])

    # Quadrant colors: high agree + high acc = green (good); high agree + low acc = red (bias)
    hmed = sub[col].median()
    amed = sub['human_acc'].median()
    colors_q = []
    for _, row in sub.iterrows():
        if   row[col] > hmed and row['human_acc'] > amed: colors_q.append('#4CAF50')  # Q1 high/high
        elif row[col] > hmed and row['human_acc'] <= amed: colors_q.append('#F44336') # Q4 high/low = BIAS
        elif row[col] <= hmed and row['human_acc'] > amed: colors_q.append('#FF9800') # Q2 low/high
        else: colors_q.append('#9E9E9E')  # Q3 low/low

    ax.scatter(sub[col], sub['human_acc'], c=colors_q, s=45, alpha=0.8, edgecolor='white', zorder=3)
    m, b = np.polyfit(sub[col], sub['human_acc'], 1)
    xline = np.linspace(sub[col].min(), sub[col].max(), 100)
    ax.plot(xline, m*xline+b, 'k-', lw=1.5, alpha=0.6)
    ax.axvline(hmed, color='gray', ls=':', lw=1)
    ax.axhline(amed, color='gray', ls=':', lw=1)

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Human accuracy (variant C)')
    ax.set_title(f'{atype.capitalize()} HH vs accuracy (n={len(sub)}q)\nr={r:.3f}, ρ={rho:.3f}, p={p:.3f}')

    # Legend
    from matplotlib.patches import Patch
    handles = [
        Patch(color='#4CAF50', label='High agree + High acc'),
        Patch(color='#F44336', label='High agree + Low acc (BIAS!)'),
        Patch(color='#FF9800', label='Low agree + High acc'),
        Patch(color='#9E9E9E', label='Low agree + Low acc'),
    ]
    ax.legend(handles=handles, fontsize=7, loc='upper left')
    ax.grid(alpha=0.2)

plt.suptitle('HH Agreement vs Human Accuracy (variant C)\nRed = systematic wrong consensus = linguistic prior exploitation',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Non-binary model answers on yes/no questions ──────────────────────────────
# Some models respond with words other than yes/no; quantify and inspect.

hm_yn = yn_pairs[(yn_pairs['variant']=='C') & (yn_pairs['pair_type']=='HM')].copy()
hm_yn['model_ans_norm'] = hm_yn['answer_2'].apply(_yn_norm)

# extract model name from subject_2
total = len(hm_yn['subject_2'].unique())
other_pairs = hm_yn[hm_yn['model_ans_norm']=='other']

print(f'HM pairs (variant C): {len(hm_yn):,}')
print(f'Model answer distribution: {hm_yn["model_ans_norm"].value_counts().to_dict()}')
print(f'Non-binary responses: {len(other_pairs):,} ({100*len(other_pairs)/len(hm_yn):.1f}%)')

if len(other_pairs) > 0:
    print('\nTop non-binary answers:')
    print(other_pairs['answer_2'].value_counts().head(15).to_string())
    print('\nModels with most non-binary responses:')
    print(other_pairs['subject_2'].value_counts().head(10).to_string())

In [ ]:
# ── Full summary table (variant C) ───────────────────────────────────────────
# Stats are mean ± std across questions (kappa/exact computed per question first,
# then aggregated — so std reflects question-level variability, not pair-level noise).

from IPython.display import display

# Yes/no summary
C_yn_sum = yn_q_long[yn_q_long['variant'] == 'C'].groupby('pair_type').agg(
    n_questions=('question_id', 'nunique'),
    n_pairs=('n_pairs', 'sum'),
    exact_mean=('exact', 'mean'),
    exact_std=('exact', 'std'),
    kappa_mean=('kappa', 'mean'),
    kappa_std=('kappa', 'std'),
    yes_rate_mean=('yes_rate', 'mean'),
    yes_rate_std=('yes_rate', 'std'),
).round(3)

print("── Yes/No Questions (variant C) — stats = mean ± std across 25 questions ──")
display(C_yn_sum)

# Free-text summary (pair-level; std is across all pairs, not per-question)
# Note: pair-level std is larger than question-level std — different interpretation
C_ft_pairs = ft_pairs[ft_pairs['variant'] == 'C']

# Compute per-question means first, then aggregate — matches yes/no methodology
ft_q_means = (C_ft_pairs.groupby(['question_id', 'pair_type'])
              [['sbert_score_clip', 'simcse_score_clip', 'exact_score']].mean())
C_ft_sum = ft_q_means.groupby('pair_type').agg(['mean', 'std']).round(3)
C_ft_sum.columns = ['_'.join(c) for c in C_ft_sum.columns]
C_ft_sum.insert(0, 'n_questions', C_ft_pairs.groupby('pair_type')['question_id'].nunique())
C_ft_sum.insert(1, 'n_pairs',     C_ft_pairs.groupby('pair_type').size())

print("\n── Free-Text Questions (variant C) — stats = mean ± std across 88 questions ──")
display(C_ft_sum)

In [ ]:
# ── Export combined table ────────────────────────────────────────────────────
out_path = EXPORTS / 'combined_agreement.parquet'
combined.to_parquet(out_path, index=False)
print(f'Saved: {len(combined)} rows → {out_path}')

# Also CSV for inspection
combined.to_csv(EXPORTS / 'combined_agreement.csv', index=False)
print(f'Saved CSV: {EXPORTS / "combined_agreement.csv"}')